# CIF -> MOFid + MOFKey + MOFJSON batch conversion

In [1]:
import sys
import json
import time
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "mofjson").is_dir():
            return candidate
    raise RuntimeError(f"no src/mofjson found above {start}")


PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

from mofid_mofjson_workers import run_mofid_on_cif, run_mofjson_on_cif

MOFID_ASSETS_DIR = Path.cwd() / "mofid_web_assets"


In [2]:
CIF_DIR = Path("/Users/yong/projects/MoFJSON/dataset/CoreMOFDB/CR/")          # <-- edit this
OUTPUT_DIR = Path.cwd() / "mof_conversion_out"
MOFJSON_DIR = OUTPUT_DIR / "mofjson_data/"
RESULTS_JSONL = OUTPUT_DIR / "results.jsonl"   # resumable working file
RESULTS_TABLE = OUTPUT_DIR / "mof_conversion.tsv"

MOFID_TIMEOUT_SECONDS = 120
MOFJSON_TIMEOUT_SECONDS = 120

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MOFJSON_DIR.mkdir(parents=True, exist_ok=True)

cif_paths = sorted(CIF_DIR.glob("*.cif"))
print(f"{len(cif_paths)} CIFs found in {CIF_DIR}")


2737 CIFs found in /Users/yong/projects/MoFJSON/dataset/CoreMOFDB/CR


In [3]:
# resumable: skip filenames already in results.jsonl
done = set()
if RESULTS_JSONL.exists():
    with open(RESULTS_JSONL) as f:
        for line in f:
            if line.strip():
                done.add(json.loads(line)["cif_filename"])
print(f"{len(done)} already converted, skipping those")


2737 already converted, skipping those


In [4]:
todo = [p for p in cif_paths if p.name not in done]
t0 = time.time()

with open(RESULTS_JSONL, "a") as out:
    for i, cif_path in enumerate(todo):
        row = {"cif_filename": cif_path.name}

        mofid_res = run_mofid_on_cif(str(cif_path), str(MOFID_ASSETS_DIR), MOFID_TIMEOUT_SECONDS)
        if mofid_res["status"] == "ok":
            row["mofid"] = mofid_res["result"]["mofid"]
            row["mofkey"] = mofid_res["result"]["mofkey"]
        else:
            row["mofid"] = None
            row["mofkey"] = None
            row["mofid_error"] = mofid_res["error"]

        mofjson_path = MOFJSON_DIR / (cif_path.stem + ".mofjson.json")
        mofjson_res = run_mofjson_on_cif(str(cif_path), str(PROJECT_ROOT), str(mofjson_path), MOFJSON_TIMEOUT_SECONDS)
        if mofjson_res["status"] == "ok":
            row["mofjson_path"] = str(mofjson_path)
        else:
            row["mofjson_path"] = None
            row["mofjson_error"] = mofjson_res["error"]

        out.write(json.dumps(row) + "\n")
        out.flush()

        if (i + 1) % 25 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            remaining = (len(todo) - (i + 1)) / rate
            print(f"[{i + 1}/{len(todo)}] ({elapsed:.0f}s elapsed, ~{remaining/60:.0f} min left)")

print(f"done: {len(todo)} converted this run")


[17:18:26] Explicit valence for atom # 68 C, 5, is greater than permitted
[17:18:26] Explicit valence for atom # 68 C, 5, is greater than permitted
[17:18:26] Explicit valence for atom # 68 C, 5, is greater than permitted
[17:18:29] Explicit valence for atom # 68 C, 5, is greater than permitted
[17:18:29] Explicit valence for atom # 68 C, 5, is greater than permitted
[17:18:29] Explicit valence for atom # 68 C, 5, is greater than permitted


[25/2737] (42s elapsed, ~77 min left)


[17:19:05] Explicit valence for atom # 11 C, 5, is greater than permitted
[17:19:05] Explicit valence for atom # 11 C, 5, is greater than permitted
[17:19:05] Explicit valence for atom # 11 C, 5, is greater than permitted
[17:19:05] Explicit valence for atom # 11 C, 5, is greater than permitted
[17:19:06] Explicit valence for atom # 11 C, 5, is greater than permitted
[17:19:06] Explicit valence for atom # 11 C, 5, is greater than permitted
[17:19:06] Explicit valence for atom # 11 C, 5, is greater than permitted
[17:19:06] Explicit valence for atom # 11 C, 5, is greater than permitted
[17:19:33] Explicit valence for atom # 9 C, 5, is greater than permitted
[17:19:33] Explicit valence for atom # 9 C, 5, is greater than permitted
[17:19:33] Explicit valence for atom # 9 C, 5, is greater than permitted
[17:19:33] Explicit valence for atom # 9 C, 5, is greater than permitted


[50/2737] (79s elapsed, ~70 min left)


[17:19:39] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:39] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:39] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:39] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:39] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:39] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:39] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:39] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:40] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:40] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:40] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:40] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:40] Explicit valence for atom # 22 C, 5, is greater than permitted
[17:19:40] Explicit valence for atom #

...

[2700/2737] (6906s elapsed, ~2 min left)


[19:13:21] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:21] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:21] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:21] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:25] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:25] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:25] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:25] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:28] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:28] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:28] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:28] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:32] Explicit valence for atom # 10 C, 5, is greater than permitted
[19:13:32] Explicit valence for atom #

[2725/2737] (6972s elapsed, ~1 min left)


[19:14:33] Explicit valence for atom # 8 C, 5, is greater than permitted
[19:14:33] Explicit valence for atom # 8 C, 5, is greater than permitted


done: 2737 converted this run


Build the final table -- 4 columns: filename, mofid, mofkey, mofjson (path to the schema file; the schema itself is too large to inline per-row for 2000+ structures).

In [4]:
import pandas as pd

rows = [json.loads(l) for l in open(RESULTS_JSONL) if l.strip()]
df = pd.DataFrame(rows)[["cif_filename", "mofid", "mofkey", "mofjson_path"]]
df.to_csv(RESULTS_TABLE, sep="\t", index=False)
print(f"wrote {len(df)} rows to {RESULTS_TABLE}")
df.head()

wrote 2737 rows to /Users/yong/projects/MoFJSON/dataset/mof_conversion_out/mof_conversion.tsv


,cif_filename,mofid,mofkey,mofjson_path
0,41467_2015_BFncomms7350_MOESM943_ESM_1_ASR_pac...,CC1=NN=C([N]1)CC1=[N]=C(N=N1)C.CC1=[N]=C(N=N1)...,Cu.BDWGWOFRNBKPSU.MOFkey-v1.UNKNOWN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
1,41467_2015_BFncomms7350_MOESM943_ESM_1_FSR_pac...,CC1=NN=C([N]1)CC1=[N]=C(N=N1)C.CC1=[N]=C(N=N1)...,Cu.BDWGWOFRNBKPSU.MOFkey-v1.UNKNOWN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
2,41467_2015_BFncomms7350_MOESM943_ESM_3_FSR_pac...,CC1=NN=C([N]1)CC1=[N]=C(N=N1)C.CC1=[N]=C(N=N1)...,Cu.BDWGWOFRNBKPSU.MOFkey-v1.UNKNOWN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
3,41467_2015_BFncomms7350_MOESM943_ESM_4_ASR_pac...,CC1=NN=C([N]1)CC1=[N]=C(N=N1)C.CC1=[N]=C(N=N1)...,Cu.BDWGWOFRNBKPSU.MOFkey-v1.UNKNOWN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
4,41467_2015_BFncomms7350_MOESM943_ESM_4_FSR_pac...,CC1=NN=C([N]1)CC1=[N]=C(N=N1)C.CC1=[N]=C(N=N1)...,Cu.BDWGWOFRNBKPSU.MOFkey-v1.UNKNOWN,/Users/yong/projects/MoFJSON/dataset/mof_conve...


In [7]:
df_tsv = pd.read_csv(RESULTS_TABLE, sep="\t")

cols = ["cif_filename", "mofid", "mofkey", "mofjson_path"]

print(f"Total Rows: {len(df_tsv)} (Expected: 2737)")
print("-" * 40)

# Check missing values for each representation
print("Missing / Null values per representation:")
print(df_tsv[cols].isna().sum())
print("-" * 40)

# Count unique values for each representation
print("Unique count per representation column:")
for col in cols:
    print(f"  • {col}: {df_tsv[col].nunique()} unique entries")

Total Rows: 2737 (Expected: 2737)
----------------------------------------
Missing / Null values per representation:
cif_filename     0
mofid           45
mofkey          45
mofjson_path     1
dtype: int64
----------------------------------------
Unique count per representation column:
  • cif_filename: 2737 unique entries
  • mofid: 2692 unique entries
  • mofkey: 1247 unique entries
  • mofjson_path: 2736 unique entries


In [8]:
# Filter for any row containing a NaN/None in the representation columns
missing_rows = df_tsv[df_tsv[["mofid", "mofkey", "mofjson_path"]].isna().any(axis=1)]

print(f"Found {len(missing_rows)} rows with missing data:")
missing_rows[["cif_filename", "mofid", "mofkey", "mofjson_path"]]

Found 46 rows with missing data:


,cif_filename,mofid,mofkey,mofjson_path
188,c5cc08456c2_1_ASR_pacman.cif,NaN,NaN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
189,c5cc08456c2_1_FSR_pacman.cif,NaN,NaN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
348,c6qi00526h2_ASR_pacman.cif,NaN,NaN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
349,c6qi00526h2_FSR_pacman.cif,NaN,NaN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
589,c7sc04266c2_ASR_pacman.cif,NaN,NaN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
590,c7sc04266c2_FSR_pacman.cif,NaN,NaN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
668,c8ce00692j2_7_FSR_pacman.cif,O=C(C1=CC=CC(=[C]1)c1ccc(c(c1)C(=O)[O-])C(=O)O...,Ni.LEOSWPQCDMBCNF.MWVTWFVJZLCBMC.NBAUUNCGSMAPF...,NaN
988,c9ce01567a2_1_ASR_pacman.cif,NaN,NaN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
989,c9ce01567a2_3_ASR_pacman.cif,NaN,NaN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
1049,c9dt02820j2_FSR_pacman.cif,NaN,NaN,/Users/yong/projects/MoFJSON/dataset/mof_conve...
